# The MLX framework for MacOS

Your MacBook can run PyTorch and Jax on CPU of course, but you miss out on GPU hardware acceleration. [MLX is the solution...](https://github.com/ml-explore/mlx)

In 2026 we also got [`mlx_lm`](https://github.com/ml-explore/mlx-lm) — basically folding in `transformers` and allowing you to easily run accelerated local models.

## Load a model

In [1]:
from mlx_lm import load

# Model ID on Hugging Face.
# This model is ungated; you need an HF token for gated models.
model_id = "Qwen/Qwen3-1.7B"

# Load the model and tokenizer simultaneously. MLX handles dtype etc.
model, tokenizer = load(model_id)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

## Submit a prompt

In [2]:
from mlx_lm import generate

generate(model, tokenizer, prompt="Once upon a time")

', in a small village, there was a problem that needed to be solved. The problem was: "A man is 3 times as old as his son. The man\'s age is 30 years more than the son\'s age. How old are they?" The answer to this problem is 45 and 15. But the problem is that the answer is not correct. The man is 3 times as old as his son, but the man\'s age is 30 years more than the son\'s age. So, the problem is not correct. But the problem is not the problem. The problem is the answer. The answer is not correct. But the problem is not the problem. The problem is the answer. The answer is not correct. But the problem is not the problem. The problem is the answer. The answer is not correct. But the problem is not the problem. The problem is the answer. The answer is not correct. But the problem is not the problem. The problem is the answer. The answer is not correct. But the problem is not the problem. The problem is the answer. The answer is not correct. But the problem is not the problem. The proble

OK, this model does not always do well with straight generation. It needs a template to push it into 'instruction' mode:

In [3]:
p = "You are a helpful assistant."
q = "Continue this story: 'Once upon a time...'"

messages = [
    {"role": "system", "content": p},
    {"role": "user", "content": q},
]

prompt_text = tokenizer._tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,  # Try changing this to True.
)

In [4]:
from mlx_lm import generate

generate(model, tokenizer, prompt=prompt_text)

'<think>\nOkay, the user wants me to continue the story starting with "Once upon a time...". Let me think about how to approach this. The original prompt is pretty open-ended, so I need to create a compelling narrative that\'s engaging and has a clear direction.\n\nFirst, I should decide on the genre. Since the user didn\'t specify, maybe a fantasy or fairy tale style would be safe. Let\'s go with a fantasy setting. The opening line is classic, so the continuation should build on that.\n\nI need to introduce a protagonist. Maybe a young girl who\'s curious and brave. Let\'s say her name is Lila. She\'s exploring a mysterious forest. The forest could be enchanted, with magical elements. Maybe there\'s a hidden kingdom or a magical creature.\n\nThe story should have a problem or a challenge. Perhaps Lila discovers a hidden door or a magical artifact. The conflict could be a threat to the kingdom, like a curse or a dark force. The resolution would involve her using her courage and intelli

In [5]:
print(prompt_text)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Continue this story: 'Once upon a time...'<|im_end|>
<|im_start|>assistant



## 'Thinking'

To skip thinking, We can try to get the model to skip the generation of the 'thinking' tokens. For example, we can try adding this to the prompt in various ways: `"\n</think>\n\n"`

In [6]:
prompt_text += "\n</think>\n\n"

generate(model, tokenizer, prompt=prompt_text)

'Once upon a time, in a quiet village nestled between rolling hills and whispering woods, there lived a young girl named Lila. She was known for her curious nature and her love for exploring the mysteries of the world around her. Every morning, she would wake up before the sun had even risen and venture into the forest, where she would listen to the rustling leaves and the distant calls of birds.\n\nOne day, while wandering deeper into the woods than she had ever gone before, Lila stumbled upon an ancient, forgotten cottage. The door was slightly ajar, and a faint scent of old parchment and forgotten memories filled the air. Curious, she stepped inside, her heart racing with a mix of excitement and trepidation.\n\nInside, she found a hidden room, its walls lined with dusty books and a single, glowing lantern. At the center of the room stood a mysterious figure, cloaked in shadows. The figure was not human—Lila realized with a start as she saw the faint glow of a magical light emanating

## What does the model _really_ 'think'?
Before decoding, the final step the model takes is sampling from the 'logits' — the raw, unnormalized scores that the LLM assigns to every  token in its vocabulary (!) before converting them into probabilities.

Let's look at those:

In [7]:
p = "You are a helpful assistant. Answer the question only."
q = "Question: I have 7 apples and eat 2, how many apples do I have left?"

messages = [
    {"role": "system", "content": p},
    {"role": "user", "content": q},
]

In [8]:
prompt_text = tokenizer._tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
)

prompt_text += "\n</think>\n\n"

print(prompt_text)

<|im_start|>system
You are a helpful assistant. Answer the question only.<|im_end|>
<|im_start|>user
Question: I have 7 apples and eat 2, how many apples do I have left?<|im_end|>
<|im_start|>assistant

</think>




In [9]:
from mlx_lm import generate

generate(model, tokenizer, prompt=prompt_text)

'You have 7 - 2 = 5 apples left.'

## Inspect the logits

Every single token has an activation; we can get at these scores by passing the input to the model — not using the `generate()` function — and inspecting the last one, which contains the logit weights for the next token. (The sequence is the same length as the input because the first token had no logits.)

In [10]:
import mlx.core as mx

# Form an array of the inputs.
input_ids = mx.array(tokenizer.encode(prompt_text))[None, :]  # Shape: (1, seq_len)

# Extract logits for the next token prediction.
# Shape: [batch_size, sequence_length, vocab_size]
logits = model(input_ids)
logits.shape

(1, 46, 151936)

In [11]:
len(input_ids[0])

46

In [12]:
next_token_logits = logits[0, -1, :]

The `torch.topk()` function gives us both values and indices, but `mx.topk()` only gives values, so we need to write an MLX function to get indices as well:

In [13]:
def mlx_topk(a, k, axis=-1, largest=True):
    """
    MLX equivalent of torch.topk. Returns (values, indices).
    (Otherwise, mx.topk() only returns the values.)
    """
    if largest:
        idx = mx.argpartition(-a, kth=k - 1, axis=axis)
    else:
        idx = mx.argpartition(a, kth=k - 1, axis=axis)
        
    # Note: order within the top-k is not guaranteed to be sorted.
    indices = mx.take_along_axis(idx, mx.arange(k), axis=axis)
    values = mx.take_along_axis(a, indices, axis=axis)
    
    return values, indices

Now we can use this to get the tokens corresponding to the top `k` logits.

In [14]:
top_k = 10
top_logits, top_indices = mlx_topk(next_token_logits, k=top_k)

for score, token_id in zip(top_logits, top_indices):
    token = tokenizer.decode(token_id.item())
    print(f"Token: {token!r:<10} | Logit: {score.item():.4f}")

Token: 'You'      | Logit: 45.7500
Token: 'I'        | Logit: 38.2500
Token: '5'        | Logit: 33.0000
Token: 'To'       | Logit: 32.5000
Token: '6'        | Logit: 30.3750
Token: '4'        | Logit: 29.6250
Token: ' You'     | Logit: 29.1250
Token: 'you'      | Logit: 28.6250
Token: '8'        | Logit: 28.5000
Token: '7'        | Logit: 28.3750


Let's only look for digits:

In [15]:
import re

pattern = re.compile(r"\d")

top_k = 100
top_logits, top_indices = mlx_topk(next_token_logits, k=top_k)

for score, token_id in zip(top_logits, top_indices):
    token = tokenizer.decode(token_id.item())
    if not pattern.match(token): continue
    print(f"Token: {token!r:<10} | Logit: {score.item():.4f}")

Token: '5'        | Logit: 33.0000
Token: '6'        | Logit: 30.3750
Token: '4'        | Logit: 29.6250
Token: '8'        | Logit: 28.5000
Token: '7'        | Logit: 28.3750
Token: '3'        | Logit: 28.1250
Token: '1'        | Logit: 26.0000
Token: '2'        | Logit: 25.2500
Token: '9'        | Logit: 24.8750
Token: '0'        | Logit: 18.5000


---

©2026 Matt Hall / Equinor — Licensed CC BY / MIT, please share this work